In [ ]:
import pandas as pd 
import numpy as np

In [ ]:
df = pd.read_csv("econocausal_enriched_synthetic_dataset.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
for col in df.select_dtypes(include=['object']).columns:
    print(f"column: '{col}' ")
    print(df[col].unique())

# Exploratory Data Analysis

In [ ]:
df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="discount_offered", y="net_profit")

plt.title("Net Profit Distribution by Discount")
plt.xlabel("Discount Offered")
plt.ylabel("Net Profit")
plt.tight_layout()
plt.show()


In [ ]:
avg_revenue = df.groupby("discount_offered")["revenue"].mean()
plt.figure(figsize=(8, 5))
avg_revenue.plot(kind="bar")

plt.title("Average Revenue by Discount Level")
plt.xlabel("Discount Offered")
plt.ylabel("Average Revenue")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="age", bins=20, kde=True)

plt.title("Customer Age Distribution")
plt.xlabel("Age")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()


In [ ]:
numeric_df = df.select_dtypes(include="number")
plt.figure(figsize=(14, 10))

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x="revenue",
    y="net_profit",
    alpha=0.5
)

plt.title("Revenue vs Net Profit")
plt.xlabel("Revenue")
plt.ylabel("Net Profit")
plt.tight_layout()
plt.show()


In [ ]:
df.head(5)

In [ ]:
X = df[
[
    "age","income_bracket","urbanicity","region",
    "tenure_months","historical_spend","recency_days",
    "browsing_frequency","avg_session_duration_sec",
    "cart_abandonment_rate","support_tickets_count",
    "email_open_rate","previous_discounts_accepted",
    "previous_purchases_12m","avg_order_value",
    "website_visits_30d","mobile_app_user","loyalty_member",
    "subscription_status","preferred_channel","product_category",
    "campaign_type","day_of_week","device_type",
    "competitor_price_index","stock_availability","seasonality_index",
    "customer_satisfaction","return_rate","days_since_last_campaign","marketing_contact_count_30d"
]]
X.head(5)

In [ ]:
X.shape

In [ ]:
X.dtypes


In [ ]:
T = df["discount_offered"]
y = df["purchased"]
y.head(10)

In [ ]:
T.head(10)

In [ ]:
X.head()

## apply one-hot encoding to categorical features

In [ ]:
X = pd.get_dummies(X,
    columns=X.select_dtypes(include=["object","category"]).columns,
    drop_first=True,dtype=int
)
X.head()

In [ ]:
X.shape

In [ ]:
X.dtypes

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,T_train,T_test,y_train,y_test=train_test_split(
    X,T,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
X_train

In [ ]:
X_train.shape

In [ ]:
X_test

In [ ]:
X_test.shape

In [ ]:
T_test.shape

In [ ]:
T_train.shape

In [ ]:
y_train.shape

In [ ]:
y_test.shape

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()

X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)
X_train_scaled.shape

In [ ]:
X_test_scaled.shape

In [ ]:
pip install econml

In [ ]:
from econml.dml import CausalForestDML
from sklearn.ensemble import RandomForestRegressor,RandomForestClassifier

In [ ]:
dml_model=CausalForestDML(
    model_y=RandomForestRegressor(
        n_estimators=100,min_samples_leaf=10,random_state=2
    ),
    model_t=RandomForestClassifier(
    n_estimators=100,min_samples_leaf=10,random_state=42
),
discrete_treatment=True,
cv=3,
n_estimators=100,
random_state=42
)

In [ ]:
dml_model.fit(
    Y=np.asarray(y_train).ravel(),
    T=np.asarray(T_train).ravel(),
    X=X_train_scaled
)


In [ ]:
ite_10=dml_model.effect(
    X_test_scaled,T0=0,T1=10
)
print("Average estimated causal effect:", ite_10.mean())